In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from pt_to_api.contribs.v1 import show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def _scatter_plot_1d(numbers, suff=""):

    # 2. Create the visualization
    plt.figure(figsize=(10, 2))
    sns.stripplot(x=numbers, color='blue', alpha=0.5, jitter=True)

    plt.title('1D Clustering Visualization' + suff)
    plt.xlabel('Value')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()

In [ ]:
KERNEL_COORDINATE = "layers.2.out_12"
INPUT_LAYER_NAME = "layers.1"
R = 2
C = 3
MODE = "light"

In [ ]:
import itertools
def scatter_plot_1d(numbers, suff=""):
    # 2. Create the visualization
    plt.figure(figsize=(20, 3))
    sns.stripplot(x=numbers, color='blue', alpha=0.5, jitter=True)

    plt.title('1D Clustering Visualization' + suff)
    plt.xlabel('Value')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()


def show_72_list(xs, **kwargs):
    res = list(itertools.chain.from_iterable([tsl(x.reshape(8,3,3)) for x in xs]))
    S(res, (20, 4*len(xs)), 8, **kwargs)
    plt.show()

def show_72(x, **kwargs):
    S(tsl(x.reshape(8,3,3)), 20, 8, **kwargs)
    plt.show()

def get_receptive(y, x, ksize=3, stride=2, padding=1):
    ys = y*stride - padding
    xs = x*stride - padding
    return (ys, xs), (ys+ksize, xs+ksize)

In [ ]:
weight_objs = list(Weight.objects.filter(coordinate__startswith=KERNEL_COORDINATE, data_type="weights"))

[w.coordinate for w in weight_objs]

In [ ]:
kernel = np.stack([w.data for w in weight_objs])
S(tsl(kernel), (20,5), 8, mode=MODE)
plt.show()

In [ ]:
from tqdm import tqdm
vals = []
for sm in tqdm(SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE)):
    vals.append(sm.data[R][C])

# well its actually quite dense
# and im not sure what the cutoff is lol
# scatter_plot_1d([v for v in vals if v > 0.02 and v < 0.05])
scatter_plot_1d(vals)

In [ ]:
# vals = list(itertools.chain.from_iterable(poi_by_vals.values()))
from pt_to_api.utils import otsu_threshold

print("all")
pvals = [v for v in vals if v > 0]
nvals = [v for v in vals if v < 0]
pos_thresh = otsu_threshold(pvals)
neg_thresh = otsu_threshold(nvals)
pos_thresh, neg_thresh

In [ ]:
(y0,x0), (y1,x1) = get_receptive(R, C, 3, 2, 1)
sm_ids = []
patches = []

for sm in tqdm(SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE)):
    # act = Activation.objects.get(input=sm.input, coordinate=sm.coordinate)
    if sm.data[R][C] < pos_thresh:
        continue
    acts = Activation.objects.filter(coordinate__startswith=INPUT_LAYER_NAME, input=sm.input).order_by("coordinate")
    acts = np.stack([a.data for a in acts])
    patch = acts[:, y0:y1, x0:x1]
    sm_ids.append({"coordinate": sm.coordinate, "input": sm.input.alias})
    patches.append(patch)

In [ ]:
pw = [p*kernel for p in patches]
lin_pw = np.array([p.reshape(-1) for p in pw])
lin_pw.shape

In [ ]:
acts = Activation.objects.filter(coordinate__startswith=INPUT_LAYER_NAME, input__alias=sm_ids[0]["input"]).order_by("coordinate")
acts = np.stack([a.data for a in acts])
patch = acts[:, y0:y1, x0:x1]
S(tsl(acts), 20, 8, mode=MODE)
plt.show()


print("verify")
print(np.all(patch == patches[0]))
S([patch.reshape(8,9), patches[0].reshape(8,9)], mode=MODE)
# S(tsl(patch), 15, 8, mode=MODE)
# plt.show()
# S(tsl(patches[0]), 15, 8, mode=MODE)
plt.show()

In [ ]:
from pt_to_api import benchmark as B
from pt_to_api.benchmark import train_x as TX

In [ ]:
scaler = B.MeanPerDimGlobalStdScaler().fit(lin_pw)
scaled_pw = scaler.transform(lin_pw)
# run = B.train(scaled_pw, 6, 1e-3, init_strategy=B.SvdInitStrategy(), use_ln_term=False, baseline_epochs=2000, epochs=4000)

In [ ]:
def _mse(x, codes, comps):
    diff = x - (codes@comps)
    return (diff**2).mean()


def get_comp_scores(X, codes, components):
    main_mse = _mse(X, codes, components)
    scores = []
    for i in range(len(components)):
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(X, codes, comps)
        print(main_mse, new_mse)
        comp_score = new_mse - main_mse
        scores.append(comp_score)
    return scores

In [ ]:
# S([c.reshape(8,9) for c in ica_estimator.components_], 20, len(ica_estimator.components_), mode=MODE)
# plt.show()
# S([c.reshape(8,9) for c in svd_run.components], 20, len(svd_run.components), mode=MODE)
# plt.show()
# S([c.reshape(8,9) for c in ica_run.components], 20, len(ica_run.components), mode=MODE)
# plt.show()

In [ ]:
# idx = 17
# S([
#     scaler.inverse_transform(run.recon[idx]).reshape(8,9), 
#     scaler.inverse_transform(scaled_pw[idx]).reshape(8,9),
#     scaler.inverse_transform(ica_run.recon[idx]).reshape(8,9),
#     scaler.inverse_transform(svd_run.recon[idx]).reshape(8,9),
# ], 10, mode=MODE, ncols=4)
# plt.show()

In [ ]:
# lets do a sweep of components
losses = []
comps = []
for comp in tqdm(range(2, 6)):
    baseline_run = B.train_baseline(scaled_pw, comp, 1e-2)
    print("baseline loss", baseline_run.loss)
    svd_run = B.train(scaled_pw, comp, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=10_000, init_strategy=B.SvdInitStrategy())
    print("svd loss", svd_run.loss)
    losses.append((baseline_run.loss, svd_run.loss))
    comps.append(svd_run.components)

In [ ]:
len(comps)

In [ ]:
svd_losses = losses[:12]
ica_losses = losses[12:]

In [ ]:
# lets do a sweep of components
ica_run_comps = []
scores = [] 
for comp in tqdm(range(2, 14)):
    baseline_run = B.train_baseline(scaled_pw, comp, 1e-2)
    print("baseline loss", baseline_run.loss)
    ica_run = B.train(scaled_pw, comp, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=10_000, init_strategy=B.IcaInitStrategy(40_000))
    print("ica run loss", ica_run.loss)
    losses.append((baseline_run.loss, ica_run.loss))
    ica_run_comps.append(ica_run.components)
    scores.append(get_comp_scores(scaled_pw, ica_run.codes, ica_run.components))

In [ ]:
# lets do a sweep of components
for comp in tqdm(range(6, 14)):
    baseline_run = B.train_baseline(scaled_pw, comp, 1e-2)
    print("baseline loss", baseline_run.loss)
    svd_run = B.train(scaled_pw, comp, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=10_000, init_strategy=B.SvdInitStrategy())
    print("svd loss", svd_run.loss)
    losses.append((baseline_run.loss, svd_run.loss))
    comps.append(svd_run.components)

In [ ]:
from sklearn.decomposition import FastICA

ica_estimator = FastICA(
    n_components=4, max_iter=100_000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(scaled_pw)

In [ ]:
S([c.reshape(8,9) for c in ica_estimator.mixing_.T], 20, len(ica_estimator.mixing_.T), mode=MODE)
plt.show()

In [ ]:
run = TX.train(scaled_pw, 4, 1e-2, 3000, use_ln_term=False)

In [ ]:
import numpy as np
_, _, Vt = np.linalg.svd(scaled_pw, full_matrices=False)
svd_w = Vt[:4]

In [ ]:
S([c.reshape(8,9) for c in svd_w], 20, len(svd_w), mode=MODE)
plt.show()

In [ ]:
S([c.reshape(8,9) for c in run.components], 20, len(run.components), mode=MODE)
plt.show()

In [ ]:
run = B.train(scaled_pw, 4, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=10_000)

In [ ]:
_, _, Vt = np.linalg.svd(scaled_pw, full_matrices=False)
svd_w = Vt[:3]
ica_estimator = FastICA(
    n_components=3, max_iter=100_000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(scaled_pw)

In [ ]:

S([c.reshape(8,9) for c in svd_w], 20, len(svd_w), mode=MODE, viztype="local")
plt.show()
S([c.reshape(8,9) for c in ica_estimator.mixing_.T], 20, len(ica_estimator.mixing_.T), mode=MODE, viztype="local")
plt.show()
S([c.reshape(8,9) for c in run.components], 20, len(run.components), mode=MODE, viztype="local")
plt.show()


In [ ]:
ica_comps[4].shape

In [ ]:
for comp in ica_comps:
    print(len(comp))
    S([c.reshape(8,9) for c in comp], 20, len(comp), mode=MODE)
    plt.show()

In [ ]:
for comp in ica_run_comps:
    print(len(comp))
    S([c.reshape(8,9) for c in comp], 20, len(comp), mode=MODE)
    plt.show()

In [ ]:
# lets do a sweep of components
for comp in tqdm(range(6, 14)):
    baseline_run = B.train_baseline(scaled_pw, comp, 1e-2)
    print("baseline loss", baseline_run.loss)
    # svd_run = B.train(scaled_pw, comp, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=10_000, init_strategy=B.SvdInitStrategy())
    print("svd loss", svd_run.loss)
    losses.append((baseline_run.loss, svd_run.loss))
    comps.append(svd_run.components)

In [ ]:
run = B.train(scaled_pw, 6, 1e-2, use_ln_term=False, baseline_epochs=800, epochs=6000, init_strategy=B.StandardInitStrategy())

In [ ]:
idx = 8
S([
    scaler.inverse_transform(ica_run.recon[idx]).reshape((8,9)), scaler.inverse_transform(scaled_pw[idx]).reshape(8,9)
], mode=MODE)
plt.show()

In [ ]:
from sklearn.decomposition import FastICA

ica_estimator = FastICA(
    n_components=6, max_iter=10_000, whiten="arbitrary-variance", tol=15e-5
)
ica_estimator.fit(scaled_pw)

In [ ]:
B.show_closest_component_of_W_for_each_component(run.components, ica_run.components, )

In [ ]:
show_72_list(ica_estimator.components_, mode=MODE)